<a href="https://colab.research.google.com/github/Sechavez16/MaestriaTecnologiaCloud/blob/main/Unidad3_EntregableFormativa_AlgoritmosDeBusqueda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ================================================================
# SISTEMA DE PRIORIZACIÓN DE ATENCIÓN ESTUDIANTIL
# ================================================================
#
# Módulo: Inteligencia Artificial
# Unidad 3: Problemas de búsqueda
# Actividad formativa
#
# Descripción:
#   Implementación de un agente inteligente que utiliza el algoritmo
#   BFS (Búsqueda en Anchura) para priorizar estudiantes con riesgo
#   de deserción académica.
#
#   El código implementa el pseudocódigo estándar de BFS descrito en
#   Russell, S. J., y Norvig, P. (2022). Inteligencia artificial:
#   Un enfoque moderno (4.ª ed.). Pearson.
#
# ================================================================

import random
from collections import deque


# ================================================================
# CLASE ESTUDIANTE
# ================================================================

class Estudiante:
    """
    Representa un estudiante con sus atributos académicos.
    """

    def __init__(self, nombre, promedio, asistencia, materias_perdidas):
        self.nombre = nombre
        self.promedio = promedio
        self.asistencia = asistencia
        self.materias_perdidas = materias_perdidas
        self.riesgo = None

    def evaluar_riesgo(self):
        """
        Evalúa el nivel de riesgo de deserción.

        Criterios:
        - Promedio < 3.0: +1 punto
        - Asistencia < 70%: +1 punto
        - Materias perdidas >= 3: +1 punto
        """
        puntos = 0

        if self.promedio < 3.0:
            puntos += 1

        if self.asistencia < 70:
            puntos += 1

        if self.materias_perdidas >= 3:
            puntos += 1

        if puntos >= 2:
            self.riesgo = "ALTO"
        elif puntos == 1:
            self.riesgo = "MEDIO"
        else:
            self.riesgo = "BAJO"

        return self.riesgo

    def obtener_prioridad(self):
        """Retorna valor numérico de prioridad (3=ALTO, 2=MEDIO, 1=BAJO)."""
        prioridades = {"ALTO": 3, "MEDIO": 2, "BAJO": 1}
        return prioridades.get(self.riesgo, 0)

    def mostrar_informacion(self):
        return (
            f"{self.nombre} | Prom: {self.promedio:.2f} | "
            f"Asist: {self.asistencia:.1f}% | Mat.perd: {self.materias_perdidas} | "
            f"Riesgo: {self.riesgo}"
        )


# ================================================================
# GENERACIÓN DE DATOS DE PRUEBA
# ================================================================

def crear_estudiantes(cantidad=4):
    """
    Genera estudiantes con datos sintéticos para la demostración.
    """
    estudiantes = []

    for i in range(cantidad):
        # Se generan datos con variabilidad para tener diferentes niveles de riesgo
        promedio = round(random.uniform(1.5, 5.0), 2)
        asistencia = round(random.uniform(40, 100), 2)
        materias_perdidas = random.randint(0, 5)

        estudiante = Estudiante(
            nombre=f"Estudiante_{i+1}",
            promedio=promedio,
            asistencia=asistencia,
            materias_perdidas=materias_perdidas
        )

        estudiante.evaluar_riesgo()
        estudiantes.append(estudiante)

    return estudiantes


# ================================================================
# PROBLEMA DE BÚSQUEDA
# ================================================================

class ProblemaPriorizacion:
    """
    Modela el problema de priorización como un problema de búsqueda.

    Elementos del problema (Russell & Norvig, 2022):
    1. Estado inicial: Estudiantes pendientes, lista de priorizados vacía
    2. Acciones: Seleccionar un estudiante pendiente
    3. Transición: Mover estudiante de pendientes a priorizados
    4. Estado objetivo: Todos priorizados en orden correcto
    5. Costo: Cada acción tiene costo 1
    """

    def __init__(self, estudiantes):
        self.estudiantes = estudiantes
        self.estado_inicial = self._crear_estado_inicial()
        self.estado_objetivo = self._crear_estado_objetivo()
        self.contador_estados = 0

    def _crear_estado_inicial(self):
        """Estado inicial: todos los estudiantes pendientes."""
        pendientes = list(self.estudiantes)
        pendientes.sort(key=lambda e: e.nombre)
        return (tuple(pendientes), tuple())

    def _crear_estado_objetivo(self):
        """
        Estado objetivo: todos los estudiantes priorizados
        en orden de mayor a menor riesgo.
        """
        priorizados = list(self.estudiantes)
        priorizados.sort(key=lambda e: e.obtener_prioridad(), reverse=True)
        return (tuple(), tuple(priorizados))

    def es_objetivo(self, estado):
        """
        Verifica si un estado es el objetivo.

        Condiciones:
        1. No hay estudiantes pendientes
        2. Los priorizados están ordenados por prioridad (ALTO > MEDIO > BAJO)
        """
        pendientes, priorizados = estado

        if len(pendientes) > 0:
            return False

        for i in range(len(priorizados) - 1):
            if priorizados[i].obtener_prioridad() < priorizados[i+1].obtener_prioridad():
                return False

        return True

    def acciones(self, estado):
        """Acciones posibles: seleccionar cualquier estudiante pendiente."""
        pendientes, _ = estado
        return list(pendientes)

    def transicion(self, estado, accion):
        """
        Función de transición: mueve un estudiante de pendientes a priorizados.
        """
        pendientes, priorizados = estado
        pendientes_list = list(pendientes)
        priorizados_list = list(priorizados)

        if accion not in pendientes_list:
            return estado

        pendientes_list.remove(accion)
        priorizados_list.append(accion)

        self.contador_estados += 1

        return (tuple(pendientes_list), tuple(priorizados_list))

    def costo(self, estado, accion):
        """Costo unitario por cada acción."""
        return 1


# ================================================================
# ALGORITMO BFS (BÚSQUEDA EN ANCHURA)
# ================================================================

def bfs(problema):
    """
    Implementación del algoritmo BFS (Breadth-First Search).

    Basado en el pseudocódigo de Russell y Norvig (2022):

    function BFS(problema) returns solución o falla
        nodo <- NODO(ESTADO_INICIAL)
        if nodo.ESTADO == OBJETIVO then return nodo
        frontera <- cola FIFO con nodo como único elemento
        explorados <- conjunto vacío

        loop do
            if frontera está vacía then return falla
            nodo <- POP(frontera)
            agregar nodo.ESTADO a explorados
            for each acción in ACCIONES(nodo.ESTADO) do
                hijo <- NODO(TRANSICIÓN(nodo.ESTADO, acción))
                if hijo.ESTADO no está en explorados y no está en frontera then
                    if hijo.ESTADO == OBJETIVO then return hijo
                    frontera <- INSERTAR(hijo, frontera)

    Parámetros:
        problema: Instancia de ProblemaPriorizacion

    Retorna:
        tuple: (solucion, estados_explorados)
            - solucion: Lista de acciones que llevan al objetivo
            - estados_explorados: Número total de estados explorados
    """

    estado_inicial = problema.estado_inicial

    # Verificar si el estado inicial ya es el objetivo
    if problema.es_objetivo(estado_inicial):
        return [], 0

    # Inicializar frontera (cola FIFO) con el estado inicial
    frontera = deque()
    frontera.append((estado_inicial, []))

    # Conjunto de estados explorados
    explorados = set()
    explorados.add(estado_inicial)

    contador_estados = 1

    while frontera:
        estado, camino = frontera.popleft()

        # Obtener acciones posibles desde el estado actual
        acciones = problema.acciones(estado)

        for accion in acciones:
            # Aplicar la acción y generar nuevo estado
            nuevo_estado = problema.transicion(estado, accion)
            nuevo_camino = camino + [accion]
            contador_estados += 1

            # Verificar si el nuevo estado es el objetivo
            if problema.es_objetivo(nuevo_estado):
                return nuevo_camino, contador_estados

            # Si no ha sido explorado, agregar a la frontera
            if nuevo_estado not in explorados:
                explorados.add(nuevo_estado)
                frontera.append((nuevo_estado, nuevo_camino))

    # No se encontró solución
    return None, contador_estados


# ================================================================
# ANÁLISIS Y REPORTE DE RESULTADOS
# ================================================================

def analizar_resultados(estudiantes, orden_atencion, estados_explorados):
    """
    Genera un reporte detallado de los resultados.
    """
    reporte = []

    reporte.append("\n" + "=" * 70)
    reporte.append("RESULTADOS DE LA PRIORIZACION")
    reporte.append("=" * 70)

    # Estadísticas de búsqueda
    reporte.append(f"\nEstadisticas de busqueda:")
    reporte.append(f"  - Total de estudiantes: {len(estudiantes)}")
    reporte.append(f"  - Estados explorados: {estados_explorados}")

    # Distribución de riesgos
    riesgos = {"ALTO": 0, "MEDIO": 0, "BAJO": 0}
    for e in estudiantes:
        riesgos[e.riesgo] += 1

    reporte.append(f"\nDistribucion de niveles de riesgo:")
    reporte.append(f"  - ALTO: {riesgos['ALTO']} estudiantes")
    reporte.append(f"  - MEDIO: {riesgos['MEDIO']} estudiantes")
    reporte.append(f"  - BAJO: {riesgos['BAJO']} estudiantes")

    # Orden de atención priorizado
    reporte.append(f"\nOrden de atencion priorizado:")
    for i, estudiante in enumerate(orden_atencion, 1):
        reporte.append(
            f"  {i}. {estudiante.mostrar_informacion()}"
        )

    # Verificación de prioridad
    reporte.append(f"\nVerificacion de prioridad:")
    correcto = True
    for i in range(len(orden_atencion) - 1):
        if orden_atencion[i].obtener_prioridad() < orden_atencion[i+1].obtener_prioridad():
            correcto = False
            break

    if correcto:
        reporte.append("  OK: Los estudiantes estan correctamente priorizados")
    else:
        reporte.append("  ERROR: La priorizacion no es correcta")

    return "\n".join(reporte)


# ================================================================
# FUNCIÓN PRINCIPAL
# ================================================================

def ejecutar_agente(cantidad_estudiantes=4):
    """
    Función principal que ejecuta el agente completo.
    """

    print("=" * 70)
    print("AGENTE INTELIGENTE DE PRIORIZACION ESTUDIANTIL")
    print("=" * 70)

    # 1. Generar estudiantes
    print(f"\nGenerando {cantidad_estudiantes} estudiantes de prueba...")
    estudiantes = crear_estudiantes(cantidad_estudiantes)

    # 2. Mostrar estudiantes
    print("\nEstudiantes sin priorizar:")
    print("-" * 50)
    for e in estudiantes:
        print(f"  {e.mostrar_informacion()}")

    # 3. Crear problema
    print("\nCreando problema de busqueda...")
    problema = ProblemaPriorizacion(estudiantes)

    # 4. Ejecutar BFS
    print("Ejecutando BFS (Busqueda en Anchura)...")
    solucion, estados_explorados = bfs(problema)

    # 5. Mostrar resultados
    if solucion is not None:
        reporte = analizar_resultados(estudiantes, solucion, estados_explorados)
        print(reporte)
    else:
        print("\nNo se encontro solucion")

    # 6. Información del agente
    print("\n" + "=" * 70)
    print("INFORMACION DEL AGENTE")
    print("=" * 70)
    print("""
Agente: Sistema de Priorizacion de Atencion Estudiantil

Sensores (Percepcion):
  - Sistema de informacion academica (calificaciones)
  - Plataforma virtual LMS (accesos, entregas, participacion)
  - Registro de asistencia (porcentaje de inasistencias)

Procesamiento (Modelo de decision):
  - Evaluacion de riesgo basada en criterios academicos
  - Asignacion de nivel de riesgo: ALTO, MEDIO o BAJO
  - Algoritmo BFS para priorizacion

Actuadores (Salidas):
  - Lista priorizada de atencion
  - Panel de seguimiento para tutores
  - Alertas tempranas por nivel de riesgo

Tipo de agente: Agente basado en utilidad con capacidad de busqueda.
""")

    return estudiantes, solucion, estados_explorados


# ================================================================
# EJECUCIÓN
# ================================================================

if __name__ == "__main__":
    estudiantes, orden_atencion, estados_explorados = ejecutar_agente(4)

AGENTE INTELIGENTE DE PRIORIZACION ESTUDIANTIL

Generando 4 estudiantes de prueba...

Estudiantes sin priorizar:
--------------------------------------------------
  Estudiante_1 | Prom: 2.95 | Asist: 91.7% | Mat.perd: 1 | Riesgo: MEDIO
  Estudiante_2 | Prom: 1.56 | Asist: 53.0% | Mat.perd: 3 | Riesgo: ALTO
  Estudiante_3 | Prom: 3.73 | Asist: 90.6% | Mat.perd: 4 | Riesgo: MEDIO
  Estudiante_4 | Prom: 2.82 | Asist: 89.6% | Mat.perd: 1 | Riesgo: MEDIO

Creando problema de busqueda...
Ejecutando BFS (Busqueda en Anchura)...

RESULTADOS DE LA PRIORIZACION

Estadisticas de busqueda:
  - Total de estudiantes: 4
  - Estados explorados: 48

Distribucion de niveles de riesgo:
  - ALTO: 1 estudiantes
  - MEDIO: 3 estudiantes
  - BAJO: 0 estudiantes

Orden de atencion priorizado:
  1. Estudiante_2 | Prom: 1.56 | Asist: 53.0% | Mat.perd: 3 | Riesgo: ALTO
  2. Estudiante_1 | Prom: 2.95 | Asist: 91.7% | Mat.perd: 1 | Riesgo: MEDIO
  3. Estudiante_3 | Prom: 3.73 | Asist: 90.6% | Mat.perd: 4 | Riesgo